# 🎨 TEST QUERY KHÓ + ĐẾM — trên VIDEO MÀU, xuất ẢNH & VIDEO từng nguồn

Sửa lỗi **video người TRẮNG ĐEN** (không test được màu áo): notebook TỰ DÒ MÀU, chọn
video NGƯỜI có màu (grocery-store/market-square), bỏ `people-walking` (trắng đen).

Mỗi nguồn (XE · NGƯỜI · CÀ CHUA · KIỆN HÀNG) chạy **2 phần**:
1. **Nhận diện QUERY KHÓ** (màu/loại/phụ kiện, dễ→khó) → **ẢNH lưới** có box + ✅/❌ det/frame.
2. **ĐẾM qua VẠCH** (detect→ByteTrack→Smoother→LineZone) → **VIDEO** annotate + số đếm.

Xuất **đầy đủ ẢNH + VIDEO cho từng nguồn** (lưu ở `hardq_out/`). Cần **GPU T4**.
LocateAnything CHẬM → cả notebook ~10-15 phút; giảm `COUNT_FRAMES`/`N_FRAMES` nếu lâu.

In [ ]:
# Cell 1 — Cài thư viện (đúng bản notebook Kaggle dùng được)
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless -q
!pip install -q -U "transformers==4.57.1" "opencv-python-headless==4.11.0.86" \
    "Pillow==11.1.0" "decord==0.6.0" "lmdb==1.7.5" accelerate peft "supervision>=0.21" matplotlib
print("✅ Đã cài. Nếu Colab báo 'Restart runtime' → Restart rồi chạy tiếp từ Cell 2.")

In [ ]:
# Cell 2 — Imports + kiểm tra GPU
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # giảm phân mảnh VRAM (đặt TRƯỚC torch)
import re, time, glob, shutil, subprocess, urllib.request
from dataclasses import dataclass
from typing import List, Tuple, Optional
import numpy as np, cv2, torch, supervision as sv, transformers
from PIL import Image
import matplotlib.pyplot as plt

print("transformers:", transformers.__version__, "(cần 4.57.1)")
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU         :", p.name, f"{p.total_memory/1024**3:.1f}GB")

In [ ]:
# Cell 3 — parse_boxes: đổi text model → bbox (nguyên từ notebook chạy được)
NORM_SCALE = 1000
_RE_BOX = re.compile(r"<box><(\d+)><(\d+)><(\d+)><(\d+)></box>")
_RE_BRK = re.compile(r"\[\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*\]")
_RE_LOC = re.compile(r"<loc_(\d+)>")

@dataclass
class Detection:
    bbox: Tuple[int, int, int, int]
    class_name: str
    confidence: float

def _add(dets, x1, y1, x2, y2, w, h, cls, conf, scale):
    x1, x2 = int(x1/scale*w), int(x2/scale*w)
    y1, y2 = int(y1/scale*h), int(y2/scale*h)
    if x2 > x1 and y2 > y1:
        dets.append(Detection((x1, y1, x2, y2), cls, conf))

def parse_boxes(text, class_name, w, h, default_conf=0.85):
    dets = []
    for m in _RE_BOX.findall(text):
        x1, y1, x2, y2 = (int(v) for v in m)
        _add(dets, x1, y1, x2, y2, w, h, class_name, default_conf, NORM_SCALE)
    if not dets:
        for m in _RE_BRK.findall(text):
            v = [float(x) for x in m]
            sc = NORM_SCALE if max(v) > 1.5 else 1
            _add(dets, v[0], v[1], v[2], v[3], w, h, class_name, default_conf, sc)
    if not dets:
        locs = _RE_LOC.findall(text)
        for i in range(0, len(locs)-3, 4):
            x1, y1, x2, y2 = (int(v) for v in locs[i:i+4])
            _add(dets, x1, y1, x2, y2, w, h, class_name, default_conf, NORM_SCALE)
    return dets

In [ ]:
# Cell 4 — LocateAnythingDetector (NGUYÊN recipe chạy được: device_map auto + generate thường)
class LocateAnythingDetector:
    def __init__(self, model_dir, max_new_tokens=1024):
        self.model_dir = model_dir
        self.max_new_tokens = max_new_tokens
        self._loaded = False
        self.dtype = torch.float16          # T4 (Turing) không có bfloat16 kernel

    def load(self):
        from transformers import AutoTokenizer, AutoProcessor, AutoConfig, AutoModel
        t0 = time.time()
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir, trust_remote_code=True)
        self.processor = AutoProcessor.from_pretrained(self.model_dir, trust_remote_code=True)
        config = AutoConfig.from_pretrained(self.model_dir, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(
            self.model_dir, config=config, trust_remote_code=True,
            torch_dtype=self.dtype, device_map="auto", attn_implementation="sdpa")
        self.model.eval()
        self._loaded = True
        print(f"✅ Loaded in {time.time()-t0:.1f}s")
        if torch.cuda.is_available():
            print(f"   GPU Mem: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
        return self

    def _prep_input(self, v):
        if isinstance(v, np.ndarray):
            v = torch.from_numpy(v)
        if torch.is_tensor(v):
            if v.is_floating_point():
                return v.to(device=self.model.device, dtype=torch.float16)
            return v.to(self.model.device)
        return v

    def detect_pil(self, pil_image, prompt, max_new_tokens=None):
        if not self._loaded: self.load()
        if torch.cuda.is_available(): torch.cuda.empty_cache()   # dọn VRAM phân mảnh trước mỗi frame → đỡ OOM
        w, h = pil_image.size
        max_tok = max_new_tokens or self.max_new_tokens
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": f"Locate all instances of: {prompt}"}]}]
        text_prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=text_prompt, images=[pil_image], return_tensors="pt")
        inputs = {k: self._prep_input(v) for k, v in inputs.items()}
        with torch.no_grad():
            output = self.model.generate(**inputs, max_new_tokens=max_tok,
                                         do_sample=False, use_cache=True, tokenizer=self.tokenizer)
        if isinstance(output, (list, tuple)) and hasattr(output[0], "shape"):
            raw = self.tokenizer.decode(output[0], skip_special_tokens=True)
        elif hasattr(output, "shape"):
            raw = self.tokenizer.decode(output[0] if output.dim() > 1 else output, skip_special_tokens=True)
        else:
            raw = str(output)
        return parse_boxes(raw, prompt, w, h), raw

    def detect_frame(self, bgr_frame, prompt, max_new_tokens=None):
        pil = Image.fromarray(cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB))
        return self.detect_pil(pil, prompt, max_new_tokens)

In [ ]:
# Cell 5 — Nguồn video (TỰ DÒ MÀU cho bài NGƯỜI) + query khó + vạch đếm
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
os.chdir(WORK)
REPO = os.path.join(WORK, "VisionOS"); BR = "claude/locate-anything-test-suite-xwju2f"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BR,
                    "https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git", REPO], check=True)
VID = os.path.join(REPO, "VisionOS", "sample_videos")

def _dl(name):
    p = os.path.join(WORK, name)
    if not os.path.exists(p):
        try:
            urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/" + name, p)
        except Exception as e:
            print("  ⚠️ tải lỗi", name, e); return None
    return p if os.path.exists(p) else None

def mean_sat(path, n=8):
    cap = cv2.VideoCapture(path); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    s = []
    for f in np.linspace(total * 0.2, total * 0.8, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f)); ok, fr = cap.read()
        if ok: s.append(float(cv2.cvtColor(fr, cv2.COLOR_BGR2HSV)[..., 1].mean()))
    cap.release()
    return float(np.mean(s)) if s else 0.0

def is_color(path):
    return path is not None and os.path.exists(path) and mean_sat(path) > 18   # <18 ≈ trắng đen

# NGƯỜI: người-đi-bộ của supervision là TRẮNG ĐEN → chọn video NGƯỜI có MÀU đầu tiên.
print("🔎 Dò video NGƯỜI có màu…")
PPL = None
for cand in ("grocery-store.mp4", "market-square.mp4", "subway.mp4", "people-walking.mp4"):
    p = _dl(cand)
    if p is None: continue
    sat = mean_sat(p); ok = sat > 18
    print(f"   {cand:22} saturation={sat:5.1f}  {'🎨 MÀU' if ok else '⬜ trắng đen'}")
    if ok and PPL is None:
        PPL = p; print(f"   → DÙNG cho bài NGƯỜI: {cand}")
if PPL is None:
    PPL = _dl("people-walking.mp4"); print("   ⚠️ không có video màu → dùng tạm trắng đen")

VEH = _dl("vehicles-2.mp4")
TOM = os.path.join(VID, "tomatoes_sorting.mp4")
ROL = os.path.join(VID, "packages_rollers.mp4")

# Mỗi nguồn: query DỄ→KHÓ + prompt đếm + kiểu/vạch (theo %). Vạch đã khớp hướng dòng chảy.
SOURCES = [
    dict(name="XE", path=VEH,
         queries=["car", "truck", "bus", "a red car", "a white car", "a large truck"],
         count_prompt="vehicle", geom=[3.5, 93.0, 93.0, 89.0]),      # ngang gần đáy (giao lộ)
    dict(name="NGUOI", path=PPL,
         queries=["person", "a person wearing a backpack", "a person in a red shirt",
                  "a person in white", "a woman"],
         count_prompt="person", geom=[0.0, 55.0, 100.0, 55.0]),      # ngang giữa
    dict(name="CA CHUA", path=TOM,
         queries=["tomato", "object", "a red tomato", "a ripe tomato", "fruit"],
         count_prompt="tomato", geom=[0.0, 70.0, 100.0, 70.0]),      # ngang y=70
    dict(name="KIEN HANG", path=ROL,
         queries=["object", "box", "package", "carton box", "a cardboard box"],
         count_prompt="object", geom=[50.0, 0.0, 50.0, 100.0]),      # DỌC x=50 (con lăn)
]
for s in SOURCES:
    s["color"] = is_color(s["path"])
RESOLUTION = (1024, 576)   # nhỏ để đỡ OOM vision
N_FRAMES = 4               # số frame / query khi test nhận diện
COUNT_FRAMES = 48          # số frame khi ĐẾM (LA chậm → vừa phải)
MAX_NEW_TOKENS = 1024      # cell nạp model cần biến này
print("\\nNguồn test:")
for s in SOURCES:
    tag = "🎨 MÀU" if s["color"] else "⬜ TRẮNG ĐEN"
    print(f"   {tag:12} {s['name']:10} {os.path.basename(str(s['path']))} | {len(s['queries'])} query")

In [ ]:
# Cell 6 — Tải model + patch bfloat16->float16 cho T4 (nguyên notebook chạy được)
from huggingface_hub import snapshot_download
MODEL_ID = "nvidia/LocateAnything-3B"
print("📥 Downloading model... (lần đầu ~6GB)")
model_dir = snapshot_download(MODEL_ID)
mc = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules")
if os.path.exists(mc): shutil.rmtree(mc)
f = os.path.join(model_dir, "modeling_locateanything.py")
if os.path.exists(f):
    real_f = os.path.realpath(f); code_txt = open(real_f).read()
    old = "pixel_values = pixel_values.to(self.language_model.dtype)"
    if old in code_txt:
        open(real_f, "w").write(code_txt.replace(old, "pixel_values = pixel_values.to(torch.float16)  # T4 fix"))
        print("✅ Patched bfloat16 -> float16")
print(f"📁 {model_dir}\n✅ Ready!")

In [ ]:
# Cell 7 — Nạp mô hình MỘT LẦN (dọn GPU trước để tránh OOM khi chạy lại)
import gc
if "detector" in globals():
    del detector
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU trống {free/1e9:.1f}/{total/1e9:.1f} GB trước khi nạp")
    if free/1e9 < 9:
        raise SystemExit("⚠️ GPU còn <9GB (model cũ KẸT). Run ▸ Restart runtime rồi chạy lại từ Cell 2 (mỗi cell 1 lần).")
detector = LocateAnythingDetector(model_dir=model_dir, max_new_tokens=MAX_NEW_TOKENS)
detector.load()
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU trống sau khi nạp: {free/1e9:.1f}/{total/1e9:.1f} GB (cần >~3GB để chạy vision).")

In [ ]:
# Cell 8 — Helper: đổi box→sv, tracker, nhận diện lưới, ĐẾM qua vạch (vẽ + lưu video)
def dets_to_sv(dets, w, h, max_frac=0.9):
    xy, cf = [], []
    for d in dets:
        x1, y1, x2, y2 = d.bbox
        if (x2 - x1) * (y2 - y1) > max_frac * w * h:   # bỏ box CẢ KHUNG
            continue
        xy.append([float(x1), float(y1), float(x2), float(y2)]); cf.append(float(d.confidence))
    if not xy:
        return sv.Detections.empty()
    return sv.Detections(xyxy=np.array(xy), confidence=np.array(cf))

def new_tracker():
    for kw in (dict(track_activation_threshold=0.1, minimum_consecutive_frames=1, lost_track_buffer=120),
               dict(track_thresh=0.1), {}):
        try:
            return sv.ByteTrack(**kw)
        except TypeError:
            continue
    return sv.ByteTrack()

def _line_px(geom, w, h):
    x1, y1, x2, y2 = geom
    return (int(x1 / 100 * w), int(y1 / 100 * h)), (int(x2 / 100 * w), int(y2 / 100 * h))

def detect_grid(det, path, queries, reso, n):
    """Nhận diện các query → (rows[(q,det/fr,✅/❌)], shots[(title,img)])."""
    w, h = reso
    cap = cv2.VideoCapture(path); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    frames = []
    for f in np.linspace(total * 0.3, total * 0.75, n):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f)); ok, fr = cap.read()
        if ok: frames.append(cv2.resize(fr, reso))
    cap.release()
    rows, shots = [], []
    for q in queries:
        tot, best, img = 0, -1, (frames[0].copy() if frames else None)
        for fr in frames:
            dts, _ = det.detect_frame(fr, q)
            sd = dets_to_sv(dts, w, h)
            tot += len(sd)
            if len(sd) > best:
                best = len(sd); im = fr.copy()
                for b in sd.xyxy.astype(int):
                    cv2.rectangle(im, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 2)
                cv2.putText(im, q[:24], (6, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                img = im
        dpf = tot / max(len(frames), 1)
        rows.append((q, round(dpf, 1), "✅" if dpf > 0 else "❌"))
        shots.append((f"{q} -> {round(dpf, 1)}", img))
    return rows, shots

def count_line(det, path, prompt, reso, geom, max_frames, out_mp4):
    """Đếm qua vạch: detect→ByteTrack→Smoother→LineZone; VẼ + lưu video. Trả số đếm."""
    w, h = reso
    tr = new_tracker()
    try:
        sm = sv.DetectionsSmoother(length=8)
    except Exception:
        sm = None
    (sx, sy), (ex, ey) = _line_px(geom, w, h)
    line = sv.LineZone(start=sv.Point(sx, sy), end=sv.Point(ex, ey))
    vw = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h))
    cap = cv2.VideoCapture(path); i = 0; seen = set()
    while i < max_frames:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.resize(fr, reso)
        dts, _ = det.detect_frame(fr, prompt)
        sd = dets_to_sv(dts, w, h)
        sd = tr.update_with_detections(sd)
        if sm is not None:
            try: sd = sm.update_with_detections(sd)
            except Exception: pass
        if sd.tracker_id is not None:
            for t in sd.tracker_id:
                if t is not None: seen.add(int(t))
        line.trigger(sd)
        ids = list(sd.tracker_id) if sd.tracker_id is not None else [None] * len(sd)
        for b, t in zip(sd.xyxy.astype(int), ids):
            cv2.rectangle(fr, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 2)
            cv2.putText(fr, f"{prompt[:8]} #{t}", (b[0], max(12, b[1] - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        cv2.line(fr, (sx, sy), (ex, ey), (0, 255, 255), 3)
        cv2.rectangle(fr, (0, 0), (w, 30), (0, 0, 0), -1)
        cv2.putText(fr, f"IN:{line.in_count} OUT:{line.out_count} total:{line.in_count+line.out_count} "
                        f"tracks:{len(seen)} f:{i+1}", (6, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        vw.write(fr); i += 1
    vw.release(); cap.release()
    return dict(total=line.in_count + line.out_count, in_=line.in_count, out=line.out_count,
                tracks=len(seen), frames=i)
print("✅ Helper sẵn sàng.")

In [ ]:
# Cell 9 — CHẠY từng nguồn: ẢNH lưới nhận diện + VIDEO đếm (LƯU HẾT)
import math
OUTDIR = os.path.join(WORK, "hardq_out"); os.makedirs(OUTDIR, exist_ok=True)
summary = []
for s in SOURCES:
    name = s["name"]; path = s["path"]
    tag = "MÀU" if s["color"] else "TRẮNG ĐEN"
    print(f"\\n{'='*62}\\n▶ {name}  ({tag})  {os.path.basename(str(path))}")
    if path is None or not os.path.exists(path):
        print("   ❌ thiếu video, bỏ qua"); continue
    if not s["color"]:
        print("   ⚠️ Video TRẮNG ĐEN → query MÀU vô nghĩa (vẫn chạy để đối chiếu).")
    # (1) nhận diện query khó
    rows, shots = detect_grid(detector, path, s["queries"], RESOLUTION, N_FRAMES)
    print("   Nhận diện query khó:")
    for q, dpf, ok in rows:
        print(f"     {ok} {q:30} det/frame={dpf}")
    k = len(shots); cols = min(3, k) or 1; nr = max(1, math.ceil(k / cols))
    fig, axes = plt.subplots(nr, cols, figsize=(5 * cols, 3.2 * nr))
    axf = list(np.atleast_1d(axes).flat)
    for ax, (t, im) in zip(axf, shots):
        if im is not None: ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(t[:34], fontsize=8); ax.axis("off")
    for ax in axf[k:]:
        ax.axis("off")
    plt.suptitle(f"{name} - nhan dien query kho ({tag})", fontsize=11)
    grid_png = os.path.join(OUTDIR, f"{name}_grid.png".replace(" ", "_"))
    plt.tight_layout(); plt.savefig(grid_png, dpi=110, bbox_inches="tight"); plt.show()
    # (2) đếm qua vạch → video
    out_mp4 = os.path.join(OUTDIR, f"{name}_count.mp4".replace(" ", "_"))
    print(f"   Đếm '{s['count_prompt']}' qua vạch ({COUNT_FRAMES} frame)…")
    res = count_line(detector, path, s["count_prompt"], RESOLUTION, s["geom"], COUNT_FRAMES, out_mp4)
    print(f"   → total={res['total']} (IN {res['in_']}/OUT {res['out']}) tracks={res['tracks']}")
    n_ok = sum(1 for r in rows if r[2] == "✅")
    summary.append(dict(name=name, color=s["color"], nok=n_ok, nq=len(rows),
                        total=res["total"], tracks=res["tracks"], grid=grid_png, mp4=out_mp4))

print("\\n" + "=" * 62 + "\\nTỔNG HỢP (mỗi nguồn: nhận diện + đếm)\\n" + "=" * 62)
print(f"{'Nguồn':12}{'Màu':6}{'Nhận diện':12}{'Đếm(total)':>11}{'tracks':>8}")
for s in summary:
    mau = "MÀU" if s["color"] else "B&W"
    ratio = f"{s['nok']}/{s['nq']}"
    print(f"{s['name']:12}{mau:6}{ratio:12}{s['total']:>11}{s['tracks']:>8}")

In [ ]:
# Cell 10 — XEM/TẢI đầy đủ: ẢNH lưới + VIDEO đếm từng nguồn (H.264)
from IPython.display import Video, display, Markdown, Image
for s in summary:
    display(Markdown(f"## {s['name']} — nhận diện {s['nok']}/{s['nq']} query · đếm total={s['total']} "
                     f"({'MÀU' if s['color'] else 'TRẮNG ĐEN'})"))
    if os.path.exists(s["grid"]):
        display(Image(filename=s["grid"], width=760))
    mp4 = s["mp4"]; h264 = mp4.replace(".mp4", "_h264.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", mp4,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", h264], check=False)
    show = h264 if os.path.exists(h264) else mp4
    display(Video(show, embed=True, width=640))
print("\\n💾 Tất cả ảnh + video đã lưu ở:", OUTDIR, "(Kaggle: tải ở panel Output).")

### Đọc kết quả
- **Bài NGƯỜI giờ chạy trên VIDEO MÀU** (tự dò, bỏ people-walking trắng đen) → query màu
  ('a person in a red shirt', 'a person in white') mới có ý nghĩa.
- **ẢNH lưới** = mỗi query 1 frame nhiều box nhất (✅ det>0 = bắt được, ❌ = không).
- **VIDEO đếm** = detect→track→đếm qua VẠCH; xem IN/OUT/total + track-id chạy trên video.
- **det/frame=0 (❌)** thường do: tiếng Việt (dùng tiếng Anh), vật KHÔNG có trong cảnh, hoặc
  mô tả quá khó. **Đếm=0** thường do VẠCH lệch dòng đi (đổi `geom` trong Cell 5).
- Muốn nhanh hơn: giảm `N_FRAMES`/`COUNT_FRAMES` (Cell 5). OOM: hạ `RESOLUTION` xuống (896,512).